# Ei, Emily — protótipo microWakeWord em português

**Experimento, não atualização pronta do Android.** Não requer suas gravações, API key ou plano pago.

1. No Colab, escolha **Ambiente de execução → Alterar tipo → T4 GPU**, se disponível gratuitamente.
2. Escolha **Ambiente de execução → Executar tudo**. Confirme apenas a execução deste notebook.
3. No final baixe o ZIP e devolva ao desenvolvimento da Emily para avaliação.

Esta segunda rodada usa **32 perfis sintéticos (Piper + Kokoro)**, incluindo femininos e masculinos. São sete perfis brasileiros e 25 estrangeiros sintetizados com fonemas pt-br; não são 32 falantes brasileiros reais. Perfis estrangeiros podem ter sotaque ou pronúncia inadequada; as prévias permitem conferir isso. Não usamos misturas de perfis para inflar o número de vozes.

Gera também frases parecidas, menções à Emily em conversas, silêncio e ruído. A duração mínima é balanceada entre classes, e a chamada inteira fica dentro da janela de treino. Vozes de validação/teste ficam separadas das de treino; não há garantia de independência de pessoas nos perfis internos do Kokoro. Resultados sintéticos não comprovam funcionamento para qualquer pessoa. Ainda faltam conversas reais longas, integração e consumo no Android. O seu áudio do WhatsApp não está incluído e não será enviado.

O Colab gratuito pode interromper a sessão ou não oferecer GPU. Nenhum plano pago é ativado. A geração salva progresso por perfil e as etapas concluídas são reutilizadas enquanto os arquivos da sessão ainda existirem. Se o Colab apagar a máquina temporária, o progresso é perdido; não conectamos seu Google Drive automaticamente.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys
root = Path('/content/emily-wakeword')
root.mkdir(exist_ok=True)
files = json.loads("{\"lab.py\": \"\\\"\\\"\\\"Isolated PT-BR microWakeWord experiment. Never installs a model in Emily.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport hashlib\\nimport json\\nimport math\\nimport os\\nfrom pathlib import Path\\nimport subprocess\\nimport sys\\nimport urllib.parse\\nimport urllib.request\\n\\nROOT = Path(__file__).resolve().parent\\nSPLITS = (\\\"training\\\", \\\"validation\\\", \\\"testing\\\")\\n\\n\\ndef read_config(path=ROOT / \\\"voices.json\\\"):\\n    config = json.loads(Path(path).read_text())\\n    ids = [v[\\\"id\\\"] for v in config[\\\"voices\\\"]]\\n    if len(ids) != len(set(ids)):\\n        raise ValueError(\\\"Uma voz não pode aparecer em mais de uma divisão.\\\")\\n    if set(v[\\\"split\\\"] for v in config[\\\"voices\\\"]) != set(SPLITS):\\n        raise ValueError(\\\"Precisamos de vozes separadas para treino, validação e teste.\\\")\\n    for voice in config[\\\"voices\\\"]:\\n        if voice[\\\"dataset_license\\\"] not in (\\\"CC0\\\", \\\"CC-BY-4.0\\\", \\\"Apache-2.0\\\"):\\n            raise ValueError(\\\"Fonte ainda não revisada para redistribuição.\\\")\\n        if voice.get(\\\"backend\\\", \\\"piper\\\") not in (\\\"piper\\\", \\\"kokoro\\\"):\\n            raise ValueError(\\\"Sintetizador não revisado.\\\")\\n        if voice.get(\\\"backend\\\") == \\\"kokoro\\\" and voice[\\\"dataset_license\\\"] != \\\"Apache-2.0\\\":\\n            raise ValueError(\\\"Licença Kokoro inesperada.\\\")\\n    return config\\n\\n\\ndef sha256(path):\\n    h = hashlib.sha256()\\n    with Path(path).open(\\\"rb\\\") as file:\\n        for block in iter(lambda: file.read(1024 * 1024), b\\\"\\\"):\\n            h.update(block)\\n    return h.hexdigest()\\n\\n\\ndef fetch(url, path):\\n    path = Path(path)\\n    if path.exists():\\n        return\\n    path.parent.mkdir(parents=True, exist_ok=True)\\n    temporary = path.with_suffix(path.suffix + \\\".part\\\")\\n    try:\\n        with urllib.request.urlopen(url, timeout=120) as response, temporary.open(\\\"wb\\\") as file:\\n            while block := response.read(1024 * 1024):\\n                file.write(block)\\n        temporary.replace(path)\\n    finally:\\n        temporary.unlink(missing_ok=True)\\n\\n\\ndef generate(work, repetitions, smoke=False):\\n    if read_config().get('schema_version') == 2:\\n        from diversity import generate as generate_diverse\\n        return generate_diverse(work, repetitions, smoke=smoke)\\n    # Load one ONNX voice at a time: do not retain four models in RAM.\\n    import numpy as np\\n    from piper import PiperVoice, SynthesisConfig\\n    from piper.config import PiperConfig\\n    from piper.phonemize_espeak import ESPEAK_DATA_DIR\\n    import onnxruntime\\n    from scipy.io import wavfile\\n    from scipy.signal import resample_poly\\n\\n    config = read_config()\\n    work.mkdir(parents=True, exist_ok=True)\\n    manifest_path = work / \\\"samples.jsonl\\\"\\n    if manifest_path.exists():\\n        raise ValueError(\\\"Use uma pasta nova para não misturar experimentos.\\\")\\n    rng = np.random.default_rng(20261004)\\n    revision = config[\\\"voices_revision\\\"]\\n    if revision == \\\"main\\\":\\n        # Resolve mutable branch once, then download every model/card at this commit.\\n        with urllib.request.urlopen(\\\"https://huggingface.co/api/models/rhasspy/piper-voices\\\", timeout=60) as response:\\n            revision = json.load(response)[\\\"sha\\\"]\\n    config[\\\"voices_revision\\\"] = revision\\n    (work / \\\"resolved_sources.json\\\").write_text(json.dumps(config, indent=2, ensure_ascii=False))\\n    records = []\\n    for voice_spec in config[\\\"voices\\\"]:\\n        voice_dir = work / \\\"voices\\\" / voice_spec[\\\"id\\\"]\\n        base = f\\\"https://huggingface.co/rhasspy/piper-voices/resolve/{revision}/\\\" + voice_spec[\\\"path\\\"]\\n        for filename in (voice_spec[\\\"id\\\"] + \\\".onnx\\\", voice_spec[\\\"id\\\"] + \\\".onnx.json\\\", \\\"MODEL_CARD\\\"):\\n            fetch(base + \\\"/\\\" + urllib.parse.quote(filename), voice_dir / filename)\\n        model = voice_dir / (voice_spec[\\\"id\\\"] + \\\".onnx\\\")\\n        model_hash = sha256(model)\\n        options = onnxruntime.SessionOptions()\\n        options.intra_op_num_threads = 1\\n        options.inter_op_num_threads = 1\\n        voice = PiperVoice(config=PiperConfig.from_dict(json.loads(Path(str(model) + \\\".json\\\").read_text())),\\n                           session=onnxruntime.InferenceSession(str(model), sess_options=options,\\n                                                              providers=[\\\"CPUExecutionProvider\\\"]),\\n                           espeak_data_dir=ESPEAK_DATA_DIR, download_dir=voice_dir)\\n        print(\\\"Gerando:\\\", voice_spec[\\\"id\\\"], voice_spec[\\\"split\\\"], flush=True)\\n        groups = ((True, config[\\\"positive_phrases\\\"]), (False, config[\\\"negative_phrases\\\"]))\\n        for positive, phrases in groups:\\n            for phrase_index, phrase in enumerate(phrases):\\n                count = repetitions if positive else max(1, repetitions // 3)\\n                for variant in range(count):\\n                    syn = SynthesisConfig(length_scale=float(rng.uniform(.8, 1.25)),\\n                                          noise_scale=float(rng.uniform(.5, .8)),\\n                                          noise_w_scale=float(rng.uniform(.6, 1.0)))\\n                    chunks = list(voice.synthesize(phrase, syn))\\n                    audio = np.concatenate([c.audio_float_array for c in chunks])\\n                    rate = chunks[0].sample_rate\\n                    divisor = math.gcd(rate, 16000)\\n                    audio = resample_poly(audio, 16000 // divisor, rate // divisor)\\n                    # Preserve the complete call; add context for PCAN stabilization.\\n                    # No arbitrary tail truncation which could train on a command instead.\\n                    audio *= 10 ** (float(rng.uniform(-18, -1)) / 20)\\n                    audio = np.pad(audio, (int(rng.uniform(.5, 1.1) * 16000), 3200))\\n                    audio += rng.normal(0, float(rng.uniform(.0001, .002)), audio.shape)\\n                    relative = Path(\\\"audio\\\") / voice_spec[\\\"split\\\"] / (\\n                        f\\\"{voice_spec['id']}-{int(positive)}-{phrase_index}-{variant}.wav\\\")\\n                    destination = work / relative\\n                    destination.parent.mkdir(parents=True, exist_ok=True)\\n                    wavfile.write(destination, 16000, np.clip(audio * 32768, -32768, 32767).astype(np.int16))\\n                    records.append(dict(path=str(relative), voice=voice_spec[\\\"id\\\"],\\n                                        split=voice_spec[\\\"split\\\"], positive=positive, text=phrase,\\n                                        seconds=len(audio) / 16000, sha256=sha256(destination),\\n                                        model_sha256=model_hash,\\n                                        license=voice_spec[\\\"dataset_license\\\"]))\\n        del voice\\n    manifest_path.write_text(\\\"\\\".join(json.dumps(r, ensure_ascii=False) + \\\"\\\\n\\\" for r in records))\\n    audit(records)\\n    print(f\\\"Gerados {len(records)} exemplos. Apenas dados sintéticos; não é validação real.\\\")\\n\\n\\ndef audit(records):\\n    if not records:\\n        raise ValueError(\\\"Manifesto vazio.\\\")\\n    voices = {}\\n    for row in records:\\n        if row[\\\"split\\\"] not in SPLITS or row[\\\"seconds\\\"] <= 0:\\n            raise ValueError(\\\"Amostra inválida.\\\")\\n        previous = voices.setdefault(row[\\\"voice\\\"], row[\\\"split\\\"])\\n        if previous != row[\\\"split\\\"]:\\n            raise ValueError(\\\"Vazamento de voz entre treino e teste.\\\")\\n    for split in SPLITS:\\n        if {r[\\\"positive\\\"] for r in records if r[\\\"split\\\"] == split} != {True, False}:\\n            raise ValueError(f\\\"Faltam positivos ou negativos em {split}.\\\")\\n\\n\\ndef features(work):\\n    import numpy as np\\n    from scipy.io import wavfile\\n    from mmap_ninja.ragged import RaggedMmap\\n    from microwakeword.audio.audio_utils import generate_features_for_clip\\n\\n    records = [json.loads(line) for line in (work / \\\"samples.jsonl\\\").read_text().splitlines()]\\n    audit(records)\\n    for positive in (True, False):\\n        for split in SPLITS:\\n            rows = [r for r in records if r[\\\"positive\\\"] == positive and r[\\\"split\\\"] == split]\\n            out = work / \\\"features\\\" / str(int(positive)) / split / \\\"samples_mmap\\\"\\n            if out.exists():\\n                raise ValueError(\\\"Características já existem; use outra pasta.\\\")\\n            def iterator():\\n                for row in rows:\\n                    path = work / row[\\\"path\\\"]\\n                    if sha256(path) != row[\\\"sha256\\\"]:\\n                        raise ValueError(\\\"Áudio alterado depois da geração.\\\")\\n                    rate, audio = wavfile.read(path)\\n                    if rate != 16000 or audio.ndim != 1 or audio.dtype != np.int16:\\n                        raise ValueError(\\\"Esperado WAV mono PCM16 de 16 kHz.\\\")\\n                    # Reference TF microfrontend, uint16 retained: upstream scales by 0.0390625.\\n                    yield generate_features_for_clip(audio, step_ms=10, use_c=False)\\n            out.parent.mkdir(parents=True, exist_ok=True)\\n            RaggedMmap.from_generator(out_dir=str(out), sample_generator=iterator(), batch_size=8, verbose=False)\\n    print(\\\"Características do microfrontend prontas.\\\")\\n\\n\\ndef training_config(work, steps):\\n    return dict(window_step_ms=10, train_dir=str(work / \\\"trained\\\"),\\n                features=[dict(features_dir=str(work / \\\"features\\\" / str(int(positive))),\\n                               sampling_weight=1., penalty_weight=1., truth=positive,\\n                               truncation_strategy=\\\"truncate_start\\\" if positive else \\\"random\\\", type=\\\"mmap\\\")\\n                          for positive in (True, False)],\\n                training_steps=[steps], positive_class_weight=[1], negative_class_weight=[2],\\n                learning_rates=[.0003], batch_size=16, time_mask_max_size=[5], time_mask_count=[1],\\n                freq_mask_max_size=[3], freq_mask_count=[1], eval_step_interval=min(steps, 250),\\n                clip_duration_ms=2040, target_minimization=.15, minimization_metric=\\\"loss\\\",\\n                # No ambient corpus yet: average_viable_recall would always be zero.\\n                # Pinned upstream does not actually return the documented false_positive_rate metric.\\n                maximization_metric=\\\"recall\\\")\\n\\n\\ndef train(work, steps):\\n    import yaml\\n    config = training_config(work, steps)\\n    signature = dict(clip_duration_ms=config[\\\"clip_duration_ms\\\"], stride=3,\\n                     upstream_commit=read_config()[\\\"upstream_commit\\\"],\\n                     training_settings=config,\\n                     samples_sha256=sha256(work / \\\"samples.jsonl\\\"))\\n    signature_path = work / \\\"checkpoint-settings.json\\\"\\n    if (work / \\\"trained\\\").exists():\\n        if not signature_path.exists() or json.loads(signature_path.read_text()) != signature:\\n            raise ValueError(\\\"Checkpoint incompatível com este experimento; use uma pasta nova.\\\")\\n    signature_path.write_text(json.dumps(signature, indent=2))\\n    path = work / \\\"training.yaml\\\"\\n    path.write_text(yaml.safe_dump(config))\\n    command = [sys.executable, str(ROOT / \\\"run_upstream.py\\\"),\\n               \\\"--training_config\\\", str(path), \\\"--train\\\", \\\"1\\\", \\\"--restore_checkpoint\\\", \\\"1\\\",\\n               \\\"--test_tf_nonstreaming\\\", \\\"0\\\", \\\"--test_tflite_nonstreaming\\\", \\\"0\\\",\\n               \\\"--test_tflite_nonstreaming_quantized\\\", \\\"0\\\", \\\"--test_tflite_streaming\\\", \\\"0\\\",\\n               \\\"--test_tflite_streaming_quantized\\\", \\\"1\\\", \\\"--use_weights\\\", \\\"best_weights\\\",\\n               \\\"mixednet\\\", \\\"--pointwise_filters\\\", \\\"32,32,32,32\\\", \\\"--repeat_in_block\\\", \\\"1,1,1,1\\\",\\n               \\\"--mixconv_kernel_sizes\\\", \\\"[5],[7,11],[9,15],[23]\\\",\\n               \\\"--residual_connection\\\", \\\"0,0,0,0\\\", \\\"--first_conv_filters\\\", \\\"16\\\",\\n               \\\"--first_conv_kernel_size\\\", \\\"5\\\", \\\"--stride\\\", \\\"3\\\"]\\n    subprocess.run(command, check=True)\\n\\n\\ndef quantize(values, details):\\n    import numpy as np\\n    scale, zero = details[\\\"quantization\\\"]\\n    if not scale:\\n        raise ValueError(\\\"Tensor sem escala de quantização.\\\")\\n    bounds = np.iinfo(details[\\\"dtype\\\"])\\n    return np.clip(np.rint(values / scale + zero), bounds.min, bounds.max).astype(details[\\\"dtype\\\"])\\n\\n\\ndef score_clip(model_path, audio):\\n    import numpy as np\\n    from ai_edge_litert.interpreter import Interpreter\\n    from microwakeword.audio.audio_utils import generate_features_for_clip\\n\\n    # Fresh interpreter per file: streaming state must never leak between speakers.\\n    model = Interpreter(model_path=str(model_path), num_threads=1)\\n    model.allocate_tensors()\\n    input_tensor = model.get_input_details()[0]\\n    output_tensor = model.get_output_details()[0]\\n    features = generate_features_for_clip(audio, step_ms=10, use_c=False).astype(np.float32) * .0390625\\n    stride = int(input_tensor[\\\"shape\\\"][1])\\n    probabilities = []\\n    for start in range(0, len(features) - stride + 1, stride):\\n        chunk = features[start:start + stride].reshape(input_tensor[\\\"shape\\\"])\\n        if np.issubdtype(input_tensor[\\\"dtype\\\"], np.integer):\\n            chunk = quantize(chunk, input_tensor)\\n        model.set_tensor(input_tensor[\\\"index\\\"], chunk)\\n        model.invoke()\\n        value = float(model.get_tensor(output_tensor[\\\"index\\\"]).reshape(-1)[0])\\n        if np.issubdtype(output_tensor[\\\"dtype\\\"], np.integer):\\n            scale, zero = output_tensor[\\\"quantization\\\"]\\n            value = (value - zero) * scale\\n        probabilities.append(value)\\n    # Five streaming outputs = 150 ms with this model's 30 ms stride.\\n    if len(probabilities) < 5:\\n        return 0.\\n    return float(np.max(np.convolve(probabilities, np.ones(5) / 5, mode=\\\"valid\\\")))\\n\\n\\ndef summarize_scores(scores):\\n    \\\"\\\"\\\"Choose by validation only; rejecting all calls is a failed candidate.\\\"\\\"\\\"\\n    from bisect import bisect_left, bisect_right\\n    for row in scores:\\n        if not math.isfinite(row[\\\"score\\\"]) or not 0 <= row[\\\"score\\\"] <= 1:\\n            raise ValueError(\\\"Pontuação inválida na avaliação.\\\")\\n    def metrics(split, threshold):\\n        positives = [r for r in scores if r[\\\"split\\\"] == split and r[\\\"positive\\\"]]\\n        negatives = [r for r in scores if r[\\\"split\\\"] == split and not r[\\\"positive\\\"]]\\n        if not positives or not negatives:\\n            raise ValueError(f\\\"Faltam classes na avaliação: {split}\\\")\\n        return dict(threshold=threshold, recall=sum(r[\\\"score\\\"] >= threshold for r in positives) / len(positives),\\n                    false_positive_rate=sum(r[\\\"score\\\"] >= threshold for r in negatives) / len(negatives),\\n                    positive_count=len(positives), negative_count=len(negatives))\\n    # Evaluate actual validation boundaries, including saturated scores above .95.\\n    # A coarse .05 grid can incorrectly reject a separable quantized model.\\n    values = {r['score'] for r in scores if r['split'] == 'validation'}\\n    thresholds = {0., 1.} | values | {math.nextafter(value, 1.) for value in values}\\n    candidates = [metrics('validation', threshold) for threshold in sorted(thresholds)]\\n    viable = [r for r in candidates if r[\\\"false_positive_rate\\\"] <= .01]\\n    # Within the permitted FPR, maximize recall rather than preferring rejection of all calls.\\n    chosen = (min(viable, key=lambda r: (-r['recall'], r['false_positive_rate'], -r['threshold']))\\n              if viable else min(candidates, key=lambda r: (r['false_positive_rate'], -r['recall'], -r['threshold'])))\\n    testing = metrics(\\\"testing\\\", chosen[\\\"threshold\\\"])\\n    diagnostics = {}\\n    for split in (\\\"validation\\\", \\\"testing\\\"):\\n        positive = sorted(r[\\\"score\\\"] for r in scores if r[\\\"split\\\"] == split and r[\\\"positive\\\"])\\n        negative = sorted(r[\\\"score\\\"] for r in scores if r[\\\"split\\\"] == split and not r[\\\"positive\\\"])\\n        # Rank AUC, with ties counted as 0.5; independent of threshold selection.\\n        auc = sum(bisect_left(negative, p) + .5 * (bisect_right(negative, p) - bisect_left(negative, p))\\n                  for p in positive) / (len(positive) * len(negative))\\n        diagnostics[split] = dict(rank_auc=auc,\\n                                 positive_range=[positive[0], positive[-1]],\\n                                 negative_range=[negative[0], negative[-1]])\\n    rejected = chosen[\\\"recall\\\"] == 0 or testing[\\\"recall\\\"] == 0 or chosen[\\\"false_positive_rate\\\"] > .01\\n    return dict(status=\\\"rejected_synthetic_evaluation\\\" if rejected else \\\"experimental_not_for_installation\\\",\\n                evidence=\\\"synthetic_only\\\", validation=chosen, testing=testing,\\n                diagnostics=diagnostics, scores=scores,\\n                reason=\\\"Não há reconhecimento útil dentro do limite de falsos positivos.\\\" if rejected\\n                       else \\\"Resultados sintéticos não comprovam desempenho real.\\\",\\n                missing=[\\\"held-out real positive speakers\\\", \\\"long real ambient recordings\\\",\\n                         \\\"Android frontend parity, battery and latency\\\"])\\n\\n\\ndef evaluate(work):\\n    from scipy.io import wavfile\\n    import numpy as np\\n    model = work / \\\"trained/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite\\\"\\n    records = [json.loads(line) for line in (work / \\\"samples.jsonl\\\").read_text().splitlines()]\\n    audit(records)\\n    scores = []\\n    for row in records:\\n        if row[\\\"split\\\"] == \\\"training\\\":\\n            continue\\n        rate, audio = wavfile.read(work / row[\\\"path\\\"])\\n        if rate != 16000 or audio.dtype != np.int16 or audio.ndim != 1:\\n            raise ValueError(\\\"Formato de avaliação inválido.\\\")\\n        scores.append(dict(split=row[\\\"split\\\"], positive=row[\\\"positive\\\"],\\n                           voice=row[\\\"voice\\\"], native_pt=row.get('native_pt'),\\n                           background=row.get('text') == '[synthetic silence/noise]',\\n                           score=score_clip(model, audio)))\\n    report = summarize_scores(scores)\\n    threshold = report['validation']['threshold']\\n    report['per_profile'] = []\\n    for voice in sorted({row['voice'] for row in scores}):\\n        rows = [row for row in scores if row['voice'] == voice]\\n        positive = [row for row in rows if row['positive']]\\n        negative = [row for row in rows if not row['positive']]\\n        background = [row for row in negative if row['background']]\\n        report['per_profile'].append(dict(voice=voice, split=rows[0]['split'], native_pt=rows[0]['native_pt'],\\n                                         positive_count=len(positive), negative_count=len(negative),\\n                                         recall=sum(r['score'] >= threshold for r in positive) / len(positive),\\n                                         false_positive_rate=sum(r['score'] >= threshold for r in negative) / len(negative),\\n                                         background_false_accepts=sum(r['score'] >= threshold for r in background)))\\n    report.update(model_sha256=sha256(model), model_bytes=model.stat().st_size)\\n    (work / \\\"report.json\\\").write_text(json.dumps(report, indent=2))\\n    print(json.dumps({k: v for k, v in report.items() if k != \\\"scores\\\"}, indent=2))\\n\\n\\ndef main():\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\\\"stage\\\", choices=(\\\"generate\\\", \\\"features\\\", \\\"train\\\", \\\"audit\\\", \\\"evaluate\\\"))\\n    parser.add_argument(\\\"--work\\\", type=Path, default=ROOT / \\\"work\\\")\\n    parser.add_argument(\\\"--repetitions\\\", type=int, default=80)\\n    parser.add_argument(\\\"--steps\\\", type=int, default=3000)\\n    parser.add_argument(\\\"--smoke\\\", action=\\\"store_true\\\", help=\\\"Lote reduzido de geração para testar o pipeline, sem valor de validação.\\\")\\n    args = parser.parse_args()\\n    if args.repetitions < 1 or args.steps < 1:\\n        parser.error(\\\"repetições e passos precisam ser positivos\\\")\\n    work = args.work.resolve()\\n    if args.stage == \\\"generate\\\": generate(work, args.repetitions, smoke=args.smoke)\\n    elif args.stage == \\\"features\\\": features(work)\\n    elif args.stage == \\\"train\\\": train(work, args.steps)\\n    elif args.stage == \\\"evaluate\\\": evaluate(work)\\n    else:\\n        audit([json.loads(line) for line in (work / \\\"samples.jsonl\\\").read_text().splitlines()])\\n        print(\\\"Divisões e classes verificadas.\\\")\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    # Set before importing TensorFlow/NumPy; default modest CPU usage, Colab may override.\\n    for variable in (\\\"OMP_NUM_THREADS\\\", \\\"OPENBLAS_NUM_THREADS\\\", \\\"TF_NUM_INTRAOP_THREADS\\\", \\\"TF_NUM_INTEROP_THREADS\\\"):\\n        os.environ.setdefault(variable, \\\"1\\\")\\n    main()\\n\", \"diversity.py\": \"\\\"\\\"\\\"Licensed synthetic diversity; styles are not claimed to be real speakers.\\\"\\\"\\\"\\nimport json\\nimport math\\nimport hashlib\\nfrom pathlib import Path\\n\\n\\ndef read_progress(path, signature, work):\\n    from lab import sha256\\n    progress = json.loads(path.read_text()) if path.exists() else dict(signature=signature, completed=[], records=[])\\n    if progress['signature'] != signature:\\n        raise ValueError('Geração interrompida com parâmetros diferentes; use uma pasta nova.')\\n    for row in progress['records']:\\n        relative = Path(row['path'])\\n        if relative.is_absolute() or '..' in relative.parts:\\n            raise ValueError('Caminho inválido no progresso de geração.')\\n        if sha256(work / relative) != row['sha256']:\\n            raise ValueError('Áudio do progresso foi alterado; use uma pasta nova.')\\n    return progress\\n\\n\\ndef checked_asset(url, path, digest):\\n    from lab import fetch, sha256\\n    fetch(url, path)\\n    if sha256(path) != digest:\\n        raise ValueError(f'Hash de origem inesperado: {path.name}')\\n\\n\\ndef kokoro_session(ort, assets, asset_dir, options):\\n    \\\"\\\"\\\"Prefer CUDA float16; keep an explicitly tested int8 CPU fallback.\\\"\\\"\\\"\\n    if 'CUDAExecutionProvider' in ort.get_available_providers():\\n        name = 'kokoro-v1.0.fp16-gpu.onnx'\\n        checked_asset(assets['release'] + name, asset_dir / name, assets['files'][name])\\n        try:\\n            session = ort.InferenceSession(str(asset_dir / name), sess_options=options,\\n                                           providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])\\n            if 'CUDAExecutionProvider' in session.get_providers():\\n                print('Kokoro: CUDA float16 (GPU).', flush=True)\\n                return session, assets['files'][name]\\n        except Exception as error:\\n            print(f'Kokoro CUDA indisponível: {type(error).__name__}; usando CPU.', flush=True)\\n    name = 'kokoro-v1.0.int8.onnx'\\n    checked_asset(assets['release'] + name, asset_dir / name, assets['files'][name])\\n    print('Kokoro: int8 CPU; geração pode demorar mais.', flush=True)\\n    return ort.InferenceSession(str(asset_dir / name), sess_options=options,\\n                                providers=['CPUExecutionProvider']), assets['files'][name]\\n\\n\\ndef augment(audio, rng, positive):\\n    \\\"\\\"\\\"Keep the entire call inside the final training window, with real context.\\\"\\\"\\\"\\n    import numpy as np\\n    from scipy.signal import resample_poly, lfilter\\n    # Time/rate perturbation creates variants, never new speaker identities.\\n    minimum_speed = max(90, math.ceil(len(audio) / (2.0 * 16000) * 100)) if positive else 90\\n    if minimum_speed > 110:\\n        raise ValueError('Chamada sintetizada longa demais para o campo receptivo; revisar pronúncia.')\\n    speed = int(rng.integers(minimum_speed, 111))\\n    audio = resample_poly(audio, 100, speed)\\n    if positive and len(audio) > int(2.0 * 16000):\\n        raise ValueError('Chamada sintetizada longa demais para o campo receptivo; revisar pronúncia.')\\n    # Random bandwidth and weak reverberation. No hard clipping of the call.\\n    if rng.random() < .35:\\n        audio = lfilter([.3], [1, -.7], audio)\\n    if rng.random() < .4:\\n        delay = int(rng.uniform(.02, .09) * 16000)\\n        echo = np.pad(audio, (delay, 0))[:len(audio)]\\n        audio = audio + float(rng.uniform(.05, .2)) * echo\\n    audio = audio * 10 ** (float(rng.uniform(-22, -2)) / 20)\\n    tail = int(rng.uniform(.06, .15) * 16000)\\n    # Equal minimum duration in both classes; do not let silence length label them.\\n    lead = max(int(.6 * 16000), int(3.3 * 16000) - len(audio) - tail)\\n    audio = np.pad(audio, (lead, tail))\\n    noise = rng.normal(0, float(rng.uniform(.00003, .003)), audio.shape)\\n    if rng.random() < .5:\\n        noise = lfilter([.15], [1, -.85], noise)\\n    return np.clip((audio + noise) * 32768, -32768, 32767).astype(np.int16)\\n\\n\\ndef generate(work, repetitions, smoke=False):\\n    import numpy as np\\n    import onnxruntime as ort\\n    from scipy.io import wavfile\\n    from scipy.signal import resample_poly\\n    from lab import read_config, fetch, sha256, audit\\n    from piper import PiperVoice, SynthesisConfig\\n    from piper.config import PiperConfig\\n    from piper.phonemize_espeak import ESPEAK_DATA_DIR\\n    from kokoro_onnx import Kokoro\\n\\n    config = read_config()\\n    if smoke:\\n        config['negative_phrases'] = ['Emily.', 'Ei, Emília.', 'Eu estava falando com a Emily.']\\n    work.mkdir(parents=True, exist_ok=True)\\n    manifest = work / 'samples.jsonl'\\n    if manifest.exists():\\n        raise ValueError('Use uma pasta nova para não misturar experimentos.')\\n    settings = dict(config=config, repetitions=repetitions, generator_sha256=sha256(Path(__file__)))\\n    signature = hashlib.sha256(json.dumps(settings, sort_keys=True).encode()).hexdigest()\\n    progress_path = work / 'generation-progress.json'\\n    progress = read_progress(progress_path, signature, work)\\n    assets = config['kokoro_assets']\\n    asset_dir = work / 'kokoro'\\n    name = 'voices-v1.0.bin'\\n    checked_asset(assets['release'] + name, asset_dir / name, assets['files'][name])\\n    card_base = 'https://huggingface.co/hexgrad/Kokoro-82M/resolve/' + assets['model_card_revision']\\n    for name in ('README.md', 'VOICES.md'):\\n        fetch(card_base + '/' + name, asset_dir / name)\\n    config['source_card_hashes'] = {name: sha256(asset_dir / name) for name in ('README.md', 'VOICES.md')}\\n    config['evaluation_note'] = 'Foreign Kokoro styles use pt-br phonemes; not native PT speakers. No voice blending.'\\n    (work / 'resolved_sources.json').write_text(json.dumps(config, indent=2, ensure_ascii=False))\\n    records = progress['records']\\n    kokoro = None\\n    def options():\\n        op = ort.SessionOptions()\\n        op.intra_op_num_threads = 1\\n        op.inter_op_num_threads = 1\\n        return op\\n    for spec in config['voices']:\\n        voice_id = spec['id']\\n        if voice_id in progress['completed']:\\n            print('Perfil já concluído:', voice_id, flush=True)\\n            continue\\n        # Voice-local seeds make resumed generation independent of skipped voices.\\n        seed = int.from_bytes(hashlib.sha256(('20261005:' + voice_id).encode()).digest()[:8], 'little')\\n        rng = np.random.default_rng(seed)\\n        backend = spec['backend']\\n        if backend == 'piper':\\n            voice_dir = work / 'voices' / voice_id\\n            base = f\\\"https://huggingface.co/rhasspy/piper-voices/resolve/{config['voices_revision']}/{spec['path']}/\\\"\\n            for name in (voice_id + '.onnx', voice_id + '.onnx.json', 'MODEL_CARD'):\\n                fetch(base + name, voice_dir / name)\\n            path = voice_dir / (voice_id + '.onnx')\\n            piper = PiperVoice(config=PiperConfig.from_dict(json.loads(Path(str(path) + '.json').read_text())),\\n                               session=ort.InferenceSession(str(path), sess_options=options(), providers=['CPUExecutionProvider']),\\n                               espeak_data_dir=ESPEAK_DATA_DIR, download_dir=voice_dir)\\n            model_hash = sha256(path)\\n            def synth(text, speed):\\n                chunks = list(piper.synthesize(text, SynthesisConfig(length_scale=1 / speed)))\\n                return np.concatenate([c.audio_float_array for c in chunks]), chunks[0].sample_rate\\n        else:\\n            if kokoro is None:\\n                session, kokoro_hash = kokoro_session(ort, assets, asset_dir, options())\\n                kokoro = Kokoro.from_session(session, str(asset_dir / 'voices-v1.0.bin'))\\n            model_hash = kokoro_hash\\n            def synth(text, speed):\\n                return kokoro.create(text, voice=spec['style'], speed=speed, lang='pt-br')\\n        print('Gerando perfil:', voice_id, spec['split'], flush=True)\\n        # Native training voices get more examples; held-out voices stay affordable.\\n        count = repetitions if spec['split'] == 'training' else min(repetitions, 24)\\n        if spec['split'] == 'training' and spec['native_pt']:\\n            count *= 2\\n        for positive, phrases in ((True, config['positive_phrases']), (False, config['negative_phrases'])):\\n            for index, phrase in enumerate(phrases):\\n                if index % 8 == 0:\\n                    print('  Chamadas' if positive else '  Negativos', index + 1, '/', len(phrases), flush=True)\\n                n = count if positive else max(1, count // 6)\\n                bases = []\\n                # Cache up to three different syntheses; waveform augmentation is cheap.\\n                for speed in (1., 1.12, .95)[:min(3, n)]:\\n                    raw, rate = synth(phrase, speed)\\n                    if not np.isfinite(raw).all() or len(raw) < rate * .1 or np.max(np.abs(raw)) < .001:\\n                        raise ValueError(f'Síntese vazia/inválida: {voice_id}, {phrase}')\\n                    divisor = math.gcd(rate, 16000)\\n                    bases.append(resample_poly(raw, 16000 // divisor, rate // divisor))\\n                for variant in range(n):\\n                    audio = augment(bases[variant % len(bases)], rng, positive)\\n                    relative = Path('audio') / spec['split'] / f'{voice_id}-{int(positive)}-{index}-{variant}.wav'\\n                    dest = work / relative\\n                    dest.parent.mkdir(parents=True, exist_ok=True)\\n                    wavfile.write(dest, 16000, audio)\\n                    records.append(dict(path=str(relative), voice=voice_id, split=spec['split'], positive=positive,\\n                                        text=phrase, native_pt=spec['native_pt'], backend=backend,\\n                                        seconds=len(audio) / 16000, sha256=sha256(dest),\\n                                        model_sha256=model_hash, license=spec['dataset_license']))\\n        # Dedicated silence/colored-noise negatives, also included in evaluation.\\n        for index in range(max(2, count // 2)):\\n            size = int((3.3 if index % 2 else 8.) * 16000)\\n            level = (0., .0001, .002, .01)[index % 4]\\n            audio = np.clip(rng.normal(0, level, size) * 32768, -32768, 32767).astype(np.int16)\\n            relative = Path('audio') / spec['split'] / f'{voice_id}-background-{index}.wav'\\n            dest = work / relative\\n            wavfile.write(dest, 16000, audio)\\n            records.append(dict(path=str(relative), voice=voice_id, split=spec['split'], positive=False,\\n                                text='[synthetic silence/noise]', seconds=size / 16000,\\n                                sha256=sha256(dest), model_sha256=model_hash, license=spec['dataset_license']))\\n        if backend == 'piper':\\n            del piper\\n        progress['completed'].append(voice_id)\\n        progress['records'] = records\\n        temporary = progress_path.with_suffix('.part')\\n        temporary.write_text(json.dumps(progress, ensure_ascii=False))\\n        temporary.replace(progress_path)\\n    audit(records)\\n    manifest.write_text(''.join(json.dumps(r, ensure_ascii=False) + '\\\\n' for r in records))\\n    print(f'{len(records)} amostras, {len(config[\\\"voices\\\"])} perfis. Ainda é avaliação sintética.', flush=True)\\n\", \"run_upstream.py\": \"\\\"\\\"\\\"Run pinned upstream, avoiding meaningless ROC when no ambient data exists.\\\"\\\"\\\"\\nimport runpy\\n\\n\\ndef ambient_guard(original):\\n    def guarded(config, folder, processor, *args, **kwargs):\\n        ambient = kwargs.get(\\\"ambient_set\\\", \\\"testing_ambient\\\")\\n        if processor.get_mode_size(ambient) == 0:\\n            print(\\\"ROC ambiente omitida: não há gravações ambiente; não interpretar como 0 falsos/hora.\\\", flush=True)\\n            return None\\n        return original(config, folder, processor, *args, **kwargs)\\n    return guarded\\n\\n\\nif __name__ == \\\"__main__\\\":\\n    import microwakeword.test as upstream_test\\n    upstream_test.tflite_streaming_model_roc = ambient_guard(upstream_test.tflite_streaming_model_roc)\\n    runpy.run_module(\\\"microwakeword.model_train_eval\\\", run_name=\\\"__main__\\\")\\n\", \"colab_setup.py\": \"\\\"\\\"\\\"Bootstrap a separate Python 3.11 even when Colab's kernel uses 3.13+.\\\"\\\"\\\"\\nfrom pathlib import Path\\nimport json\\nimport os\\nimport shutil\\nimport subprocess\\nimport sys\\n\\n\\ndef run(command, root, label):\\n    \\\"\\\"\\\"Stream real diagnostics and retain a complete log on failure.\\\"\\\"\\\"\\n    root = Path(root)\\n    root.mkdir(parents=True, exist_ok=True)\\n    print(f\\\"\\\\n--- {label} ---\\\", flush=True)\\n    tail = []\\n    with (root / \\\"setup.log\\\").open(\\\"a\\\", encoding=\\\"utf-8\\\") as log:\\n        log.write(f\\\"\\\\n--- {label} ---\\\\n\\\")\\n        with subprocess.Popen([str(part) for part in command], cwd=root,\\n                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT,\\n                              stdin=subprocess.DEVNULL, text=True, errors=\\\"replace\\\", bufsize=1) as process:\\n            for line in process.stdout:\\n                print(line, end=\\\"\\\", flush=True)\\n                log.write(line)\\n                log.flush()\\n                tail = (tail + [line])[-30:]\\n            status = process.wait()\\n    if status:\\n        raise RuntimeError(f\\\"Falhou: {label} (código {status}). \\\"\\n                           f\\\"Log completo: {root / 'setup.log'}\\\\n\\\" + \\\"\\\".join(tail))\\n\\n\\ndef ensure_python(root):\\n    root = Path(root).resolve()\\n    bootstrap = root / \\\".bootstrap\\\"\\n    uv = bootstrap / \\\"bin/uv\\\"\\n    python = root / \\\".training-venv/bin/python\\\"\\n    if not uv.exists():\\n        run([sys.executable, \\\"-m\\\", \\\"pip\\\", \\\"install\\\", \\\"--disable-pip-version-check\\\",\\n             \\\"--target\\\", bootstrap, \\\"uv==0.8.22\\\"], root, \\\"Instalar gerenciador de Python\\\")\\n    os.environ[\\\"UV_PYTHON_INSTALL_DIR\\\"] = str(root / \\\".managed-python\\\")\\n    os.environ[\\\"UV_CACHE_DIR\\\"] = str(root / \\\".uv-cache\\\")\\n    os.environ[\\\"UV_NATIVE_TLS\\\"] = \\\"true\\\"\\n    if not python.exists():\\n        run([uv, \\\"venv\\\", \\\"--managed-python\\\", \\\"--python\\\", \\\"3.11\\\", \\\"--seed\\\", python.parent.parent],\\n            root, \\\"Criar Python 3.11 separado do kernel do Colab\\\")\\n    run([python, \\\"-c\\\", \\\"import sys; print('Python de treinamento:', sys.version); \\\"\\n         \\\"assert sys.version_info[:2] == (3, 11), 'Ambiente incompatível; use uma sessão nova.'\\\"],\\n        root, \\\"Conferir versão do treinamento\\\")\\n    return python\\n\\n\\ndef setup(root):\\n    root = Path(root).resolve()\\n    python = ensure_python(root)\\n    run([python, \\\"-m\\\", \\\"pip\\\", \\\"install\\\", \\\"--disable-pip-version-check\\\",\\n         \\\"-r\\\", \\\"requirements-training.txt\\\"], root, \\\"Instalar dependências compatíveis\\\")\\n    # CUDA packages live only in this virtual environment, not the Colab kernel.\\n    if shutil.which(\\\"nvidia-smi\\\"):\\n        run([python, \\\"-m\\\", \\\"pip\\\", \\\"install\\\", \\\"--disable-pip-version-check\\\",\\n             \\\"tensorflow[and-cuda]==2.18.1\\\"], root, \\\"Preparar bibliotecas da GPU\\\")\\n        # Only one ORT distribution may own the module. This is the isolated venv.\\n        run([python, '-m', 'pip', 'uninstall', '-y', 'onnxruntime', 'onnxruntime-gpu'],\\n            root, 'Preparar ONNX Runtime único')\\n        run([python, '-m', 'pip', 'install', 'onnxruntime-gpu==1.22.0'],\\n            root, 'Habilitar síntese Kokoro na GPU')\\n        library_root = python.parent.parent / 'lib/python3.11/site-packages/nvidia'\\n        libraries = sorted(str(p) for p in library_root.glob('*/lib') if p.is_dir())\\n        os.environ['LD_LIBRARY_PATH'] = ':'.join(libraries + [os.environ.get('LD_LIBRARY_PATH', '')])\\n    run([python, \\\"-m\\\", \\\"pip\\\", \\\"install\\\", \\\"--no-deps\\\", \\\"attrs==25.4.0\\\"],\\n        root, \\\"Compatibilidade do audio-metadata\\\")\\n    if not (root / \\\"upstream/.git\\\").exists():\\n        run([\\\"git\\\", \\\"clone\\\", \\\"https://github.com/kahrendt/microWakeWord.git\\\", \\\"upstream\\\"],\\n            root, \\\"Baixar microWakeWord\\\")\\n    commit = json.loads((root / \\\"voices.json\\\").read_text())[\\\"upstream_commit\\\"]\\n    run([\\\"git\\\", \\\"-C\\\", \\\"upstream\\\", \\\"checkout\\\", commit], root, \\\"Fixar revisão testada\\\")\\n    run([python, \\\"-m\\\", \\\"pip\\\", \\\"install\\\", \\\"--no-deps\\\", \\\"-e\\\", \\\"upstream\\\"],\\n        root, \\\"Instalar microWakeWord completo\\\")\\n    run([python, \\\"-m\\\", \\\"unittest\\\", \\\"-v\\\", \\\"test_lab\\\"], root, \\\"Testar preparação\\\")\\n    run([python, \\\"-c\\\", \\\"import tensorflow as tf; \\\"\\n         \\\"print('TensorFlow:', tf.__version__); \\\"\\n         \\\"print('GPU disponível:', tf.config.list_physical_devices('GPU'))\\\"],\\n        root, \\\"Conferir TensorFlow e GPU\\\")\\n    return python\\n\", \"voices.json\": \"{\\n  \\\"upstream_commit\\\": \\\"4665173cd35f1cff9a61e06fc427f124766c488e\\\",\\n  \\\"voices_revision\\\": \\\"c10ece1aade47bb51c153c893d14e5bf8e5b7117\\\",\\n  \\\"voices\\\": [\\n    {\\n      \\\"id\\\": \\\"pt_BR-cadu-medium\\\",\\n      \\\"path\\\": \\\"pt/pt_BR/cadu/medium\\\",\\n      \\\"split\\\": \\\"training\\\",\\n      \\\"dataset_license\\\": \\\"CC0\\\",\\n      \\\"source\\\": \\\"https://github.com/OHF-Voice/voice-datasets\\\",\\n      \\\"backend\\\": \\\"piper\\\",\\n      \\\"native_pt\\\": true\\n    },\\n    {\\n      \\\"id\\\": \\\"pt_BR-edresson-low\\\",\\n      \\\"path\\\": \\\"pt/pt_BR/edresson/low\\\",\\n      \\\"split\\\": \\\"training\\\",\\n      \\\"dataset_license\\\": \\\"CC-BY-4.0\\\",\\n      \\\"source\\\": \\\"https://github.com/Edresson/TTS-Portuguese-Corpus\\\",\\n      \\\"backend\\\": \\\"piper\\\",\\n      \\\"native_pt\\\": true\\n    },\\n    {\\n      \\\"id\\\": \\\"pt_BR-faber-medium\\\",\\n      \\\"path\\\": \\\"pt/pt_BR/faber/medium\\\",\\n      \\\"split\\\": \\\"validation\\\",\\n      \\\"dataset_license\\\": \\\"CC0\\\",\\n      \\\"source\\\": \\\"https://github.com/OHF-Voice/voice-datasets\\\",\\n      \\\"backend\\\": \\\"piper\\\",\\n      \\\"native_pt\\\": true\\n    },\\n    {\\n      \\\"id\\\": \\\"pt_BR-jeff-medium\\\",\\n      \\\"path\\\": \\\"pt/pt_BR/jeff/medium\\\",\\n      \\\"split\\\": \\\"testing\\\",\\n      \\\"dataset_license\\\": \\\"CC0\\\",\\n      \\\"source\\\": \\\"https://github.com/OHF-Voice/voice-datasets\\\",\\n      \\\"backend\\\": \\\"piper\\\",\\n      \\\"native_pt\\\": true\\n    },\\n    {\\n      \\\"id\\\": \\\"kokoro-af_alloy\\\",\\n      \\\"style\\\": \\\"af_alloy\\\",\\n      \\\"backend\\\": \\\"kokoro\\\",\\n      \\\"split\\\": \\\"training\\\",\\n      \\\"native_pt\\\": false,\\n      \\\"dataset_license\\\": \\\"Apache-2.0\\\",\\n      \\\"source\\\": \\\"https://huggingface.co/hexgrad/Kokoro-82M\\\"\\n    },\\n    {\\n      \\\"id\\\": \\\"kokoro-af_aoede\\\",\\n      \\\"style\\\": \\\"af_aoede\\\",\\n      \\\"backend\\\": \\\"kokoro\\\",\\n      \\\"split\\\": \\\"training\\\",\\n      \\\"native_pt\\\": false,\\n      \\\"dataset_license\\\": \\\"Apache-2.0\\\",\\n      \\\"source\\\": \\\"https://huggingface.co/hexgrad/Kokoro-82M\\\"\\n    },\\n    {\\n      \\\"id\\\": \\\"kokoro-af_bella\\\",\\n      \\\"style\\\": \\\"af_bella\\\",\\n      \\\"backend\\\": \\\"kokoro\\\",\\n      \\\"split\\\": \\\"training\\\",\\n      \\\"native_pt\\\": false,\\n      \\\"dataset_license\\\": \\\"Apache-2.0\\\",\\n      \\\"source\\\": \\\"https://huggingface.co/hexgrad/Kokoro-82M\\\"\\n    },\\n    {\\n      \\\"id\\\": \\\"kokoro-af_kore\\\",\\n      \\\"style\\\": \\\"af_kore\\\",\\n      \\\"backend\\\": \\\"kokoro\\\",\\n      \\\"split\\\": \\\"training\\\",\\n      \\\"native_pt\\\": false,\\n      \\\"dataset_license\\\": \\\"Apache-2.0\\\",\\n      \\\"source\\\": \\\"https://huggingface.co/hexgrad/Kokoro-82M\\\"\\n    },\\n    {\\n      \\\"id\\\": \\\"kokoro-af_nicole\\\",\\n      \\\"style\\\": \\\"af_nicole\\\",\\n      \\\"backend\\\": \\\"kokoro\\\",\\n      \\\"split\\\": \\\"testing\\\",\\n      \\\"native_pt\\\": false,\\n      \\\"dataset_license\\\": \\\"Apache-2.0\\\",\\n      \\\"source\\\": \\\"https://huggingface.co/hexgrad/Kokoro-82M\\\"\\n    },\\n    {\\n      \\\"id\\\": \\\"kokoro-af_nova\\\",\\n      \\\"style\\\": \\\"af_nova\\\",\\n      \\\"backend\\\": \\\"kokoro\\\",\\n      \\\"split\\\": \\\"training\\\",\\n      \\\"native_pt\\\": false,\\n      \\\"dataset_license\\\": \\\"Apache-2.0\\\",\\n      \\\"source\\\": \\\"https://huggingface.co/hexgrad/Kokoro-82M\\\"\\n    },\\n    {\\n      \\\"id\\\": \\\"kokoro-af_river\\\",\\n      \\\"style\\\": \\\"af_river\\\",\\n      \\\"backend\\\": \\\"kokoro\\\",\\n      \\\"split\\\": \\\"training\\\",\\n      \\\"native_pt\\\": false,\\n      \\\"dataset_license\\\": \\\"Apache-2.0\\\",\\n      \\\"source\\\": \\\"https://huggingface.co/hexgrad/Kokoro-82M\\\"\\n    },\\n    {\\n      \\\"id\\\": \\\"kokoro-af_sarah\\\",\\n      \\\"style\\\": \\\"af_sarah\\\",\\n      \\\"backend\\\": \\\"kokoro\\\",\\n      \\\"split\\\": \\\"training\\\",\\n      \\\"native_pt\\\": false,\\n      \\\"dataset_license\\\": \\\"Apache-2.0\\\",\\n      \\\"source\\\": \\\"https://huggingface.co/hexgrad/Kokoro-82M\\\"\\n    },\\n    {\\n      \\\"id\\\": \\\"kokoro-am_echo\\\",\\n      \\\"style\\\": \\\"am_echo\\\",\\n      \\\"backend\\\": \\\"kokoro\\\",\\n      \\\"split\\\": \\\"training\\\",\\n      \\\"native_pt\\\": false,\\n      \\\"dataset_license\\\": \\\"Apache-2.0\\\",\\n      \\\"source\\\": \\\"https://huggingface.co/hexgrad/Kokoro-82M\\\"\\n    },\\n    {\\n      \\\"id\\\": \\\"kokoro-am_eric\\\",\\n      \\\"style\\\": \\\"am_eric\\\",\\n      \\\"backend\\\": \\\"kokoro\\\",\\n      \\\"split\\\": \\\"training\\\",\\n      \\\"native_pt\\\": false,\\n      \\\"dataset_license\\\": \\\"Apache-2.0\\\",\\n      \\\"source\\\": \\\"https://huggingface.co/hexgrad/Kokoro-82M\\\"\\n    },\\n    {\\n      \\\"id\\\": \\\"kokoro-am_fenrir\\\",\\n      \\\"style\\\": \\\"am_fenrir\\\",\\n      \\\"backend\\\": \\\"kokoro\\\",\\n      \\\"split\\\": \\\"training\\\",\\n      \\\"native_pt\\\": false,\\n      \\\"dataset_license\\\": \\\"Apache-2.0\\\",\\n      \\\"source\\\": \\\"https://huggingface.co/hexgrad/Kokoro-82M\\\"\\n    },\\n    {\\n      \\\"id\\\": \\\"kokoro-am_liam\\\",\\n      \\\"style\\\": \\\"am_liam\\\",\\n      \\\"backend\\\": \\\"kokoro\\\",\\n      \\\"split\\\": \\\"training\\\",\\n      \\\"native_pt\\\": false,\\n      \\\"dataset_license\\\": \\\"Apache-2.0\\\",\\n      \\\"source\\\": \\\"https://huggingface.co/hexgrad/Kokoro-82M\\\"\\n    },\\n    {\\n      \\\"id\\\": \\\"kokoro-am_michael\\\",\\n      \\\"style\\\": \\\"am_michael\\\",\\n      \\\"backend\\\": \\\"kokoro\\\",\\n      \\\"split\\\": \\\"testing\\\",\\n      \\\"native_pt\\\": false,\\n      \\\"dataset_license\\\": \\\"Apache-2.0\\\",\\n      \\\"source\\\": \\\"https://huggingface.co/hexgrad/Kokoro-82M\\\"\\n    },\\n    {\\n      \\\"id\\\": \\\"kokoro-am_onyx\\\",\\n      \\\"style\\\": \\\"am_onyx\\\",\\n      \\\"backend\\\": \\\"kokoro\\\",\\n      \\\"split\\\": \\\"training\\\",\\n      \\\"native_pt\\\": false,\\n      \\\"dataset_license\\\": \\\"Apache-2.0\\\",\\n      \\\"source\\\": \\\"https://huggingface.co/hexgrad/Kokoro-82M\\\"\\n    },\\n    {\\n      \\\"id\\\": \\\"kokoro-am_puck\\\",\\n      \\\"style\\\": \\\"am_puck\\\",\\n      \\\"backend\\\": \\\"kokoro\\\",\\n      \\\"split\\\": \\\"training\\\",\\n      \\\"native_pt\\\": false,\\n      \\\"dataset_license\\\": \\\"Apache-2.0\\\",\\n      \\\"source\\\": \\\"https://huggingface.co/hexgrad/Kokoro-82M\\\"\\n    },\\n    {\\n      \\\"id\\\": \\\"kokoro-bf_alice\\\",\\n      \\\"style\\\": \\\"bf_alice\\\",\\n      \\\"backend\\\": \\\"kokoro\\\",\\n      \\\"split\\\": \\\"training\\\",\\n      \\\"native_pt\\\": false,\\n      \\\"dataset_license\\\": \\\"Apache-2.0\\\",\\n      \\\"source\\\": \\\"https://huggingface.co/hexgrad/Kokoro-82M\\\"\\n    },\\n    {\\n      \\\"id\\\": \\\"kokoro-bf_emma\\\",\\n      \\\"style\\\": \\\"bf_emma\\\",\\n      \\\"backend\\\": \\\"kokoro\\\",\\n      \\\"split\\\": \\\"validation\\\",\\n      \\\"native_pt\\\": false,\\n      \\\"dataset_license\\\": \\\"Apache-2.0\\\",\\n      \\\"source\\\": \\\"https://huggingface.co/hexgrad/Kokoro-82M\\\"\\n    },\\n    {\\n      \\\"id\\\": \\\"kokoro-bf_isabella\\\",\\n      \\\"style\\\": \\\"bf_isabella\\\",\\n      \\\"backend\\\": \\\"kokoro\\\",\\n      \\\"split\\\": \\\"training\\\",\\n      \\\"native_pt\\\": false,\\n      \\\"dataset_license\\\": \\\"Apache-2.0\\\",\\n      \\\"source\\\": \\\"https://huggingface.co/hexgrad/Kokoro-82M\\\"\\n    },\\n    {\\n      \\\"id\\\": \\\"kokoro-bf_lily\\\",\\n      \\\"style\\\": \\\"bf_lily\\\",\\n      \\\"backend\\\": \\\"kokoro\\\",\\n      \\\"split\\\": \\\"training\\\",\\n      \\\"native_pt\\\": false,\\n      \\\"dataset_license\\\": \\\"Apache-2.0\\\",\\n      \\\"source\\\": \\\"https://huggingface.co/hexgrad/Kokoro-82M\\\"\\n    },\\n    {\\n      \\\"id\\\": \\\"kokoro-bm_daniel\\\",\\n      \\\"style\\\": \\\"bm_daniel\\\",\\n      \\\"backend\\\": \\\"kokoro\\\",\\n      \\\"split\\\": \\\"training\\\",\\n      \\\"native_pt\\\": false,\\n      \\\"dataset_license\\\": \\\"Apache-2.0\\\",\\n      \\\"source\\\": \\\"https://huggingface.co/hexgrad/Kokoro-82M\\\"\\n    },\\n    {\\n      \\\"id\\\": \\\"kokoro-bm_fable\\\",\\n      \\\"style\\\": \\\"bm_fable\\\",\\n      \\\"backend\\\": \\\"kokoro\\\",\\n      \\\"split\\\": \\\"training\\\",\\n      \\\"native_pt\\\": false,\\n      \\\"dataset_license\\\": \\\"Apache-2.0\\\",\\n      \\\"source\\\": \\\"https://huggingface.co/hexgrad/Kokoro-82M\\\"\\n    },\\n    {\\n      \\\"id\\\": \\\"kokoro-bm_george\\\",\\n      \\\"style\\\": \\\"bm_george\\\",\\n      \\\"backend\\\": \\\"kokoro\\\",\\n      \\\"split\\\": \\\"training\\\",\\n      \\\"native_pt\\\": false,\\n      \\\"dataset_license\\\": \\\"Apache-2.0\\\",\\n      \\\"source\\\": \\\"https://huggingface.co/hexgrad/Kokoro-82M\\\"\\n    },\\n    {\\n      \\\"id\\\": \\\"kokoro-bm_lewis\\\",\\n      \\\"style\\\": \\\"bm_lewis\\\",\\n      \\\"backend\\\": \\\"kokoro\\\",\\n      \\\"split\\\": \\\"validation\\\",\\n      \\\"native_pt\\\": false,\\n      \\\"dataset_license\\\": \\\"Apache-2.0\\\",\\n      \\\"source\\\": \\\"https://huggingface.co/hexgrad/Kokoro-82M\\\"\\n    },\\n    {\\n      \\\"id\\\": \\\"kokoro-pf_dora\\\",\\n      \\\"style\\\": \\\"pf_dora\\\",\\n      \\\"backend\\\": \\\"kokoro\\\",\\n      \\\"split\\\": \\\"training\\\",\\n      \\\"native_pt\\\": true,\\n      \\\"dataset_license\\\": \\\"Apache-2.0\\\",\\n      \\\"source\\\": \\\"https://huggingface.co/hexgrad/Kokoro-82M\\\"\\n    },\\n    {\\n      \\\"id\\\": \\\"kokoro-pm_alex\\\",\\n      \\\"style\\\": \\\"pm_alex\\\",\\n      \\\"backend\\\": \\\"kokoro\\\",\\n      \\\"split\\\": \\\"training\\\",\\n      \\\"native_pt\\\": true,\\n      \\\"dataset_license\\\": \\\"Apache-2.0\\\",\\n      \\\"source\\\": \\\"https://huggingface.co/hexgrad/Kokoro-82M\\\"\\n    },\\n    {\\n      \\\"id\\\": \\\"kokoro-pm_santa\\\",\\n      \\\"style\\\": \\\"pm_santa\\\",\\n      \\\"backend\\\": \\\"kokoro\\\",\\n      \\\"split\\\": \\\"testing\\\",\\n      \\\"native_pt\\\": true,\\n      \\\"dataset_license\\\": \\\"Apache-2.0\\\",\\n      \\\"source\\\": \\\"https://huggingface.co/hexgrad/Kokoro-82M\\\"\\n    },\\n    {\\n      \\\"id\\\": \\\"kokoro-if_sara\\\",\\n      \\\"style\\\": \\\"if_sara\\\",\\n      \\\"backend\\\": \\\"kokoro\\\",\\n      \\\"split\\\": \\\"training\\\",\\n      \\\"native_pt\\\": false,\\n      \\\"dataset_license\\\": \\\"Apache-2.0\\\",\\n      \\\"source\\\": \\\"https://huggingface.co/hexgrad/Kokoro-82M\\\"\\n    },\\n    {\\n      \\\"id\\\": \\\"kokoro-im_nicola\\\",\\n      \\\"style\\\": \\\"im_nicola\\\",\\n      \\\"backend\\\": \\\"kokoro\\\",\\n      \\\"split\\\": \\\"training\\\",\\n      \\\"native_pt\\\": false,\\n      \\\"dataset_license\\\": \\\"Apache-2.0\\\",\\n      \\\"source\\\": \\\"https://huggingface.co/hexgrad/Kokoro-82M\\\"\\n    }\\n  ],\\n  \\\"positive_phrases\\\": [\\n    \\\"Ei, Emily.\\\",\\n    \\\"Ei, Emili.\\\",\\n    \\\"Ei, Êmili.\\\"\\n  ],\\n  \\\"negative_phrases\\\": [\\n    \\\"Emily.\\\",\\n    \\\"A Emily é uma assistente.\\\",\\n    \\\"Eu estava falando com a Emily.\\\",\\n    \\\"E me liga amanhã.\\\",\\n    \\\"Ei, amigo.\\\",\\n    \\\"Ei, me lembra de comprar pão.\\\",\\n    \\\"Ele me disse que estava tudo bem.\\\",\\n    \\\"Eu queria falar com você.\\\",\\n    \\\"Abre o YouTube.\\\",\\n    \\\"E aí, como você está?\\\",\\n    \\\"É melhor esperar.\\\",\\n    \\\"A família chegou.\\\",\\n    \\\"Eu li uma mensagem.\\\",\\n    \\\"Me explica isso.\\\",\\n    \\\"Ei, Siri.\\\",\\n    \\\"Ei, Google.\\\",\\n    \\\"Ei, Emílio.\\\",\\n    \\\"Ei, Emília.\\\",\\n    \\\"Ei, Elisa.\\\",\\n    \\\"Ei, amigo, me liga.\\\",\\n    \\\"E a Emily chegou?\\\",\\n    \\\"A Emily falou comigo ontem.\\\",\\n    \\\"Emily, você está aí?\\\",\\n    \\\"Eu vi a Emily na rua.\\\",\\n    \\\"E me livrei disso.\\\",\\n    \\\"Ele me ligou hoje.\\\",\\n    \\\"E aí, minha filha?\\\",\\n    \\\"É incrível.\\\",\\n    \\\"Nem me lembro.\\\",\\n    \\\"Tem milho na cozinha.\\\",\\n    \\\"Eu queria falar da Emily, mas depois resolvo.\\\",\\n    \\\"Hoje eu fui ao mercado e encontrei minha família.\\\",\\n    \\\"Você sabe onde estão as chaves? Vamos sair daqui a pouco.\\\",\\n    \\\"Meu telefone está tocando, eu vou atender.\\\",\\n    \\\"Nós estávamos conversando sobre música, filmes e trabalho.\\\",\\n    \\\"Boa noite, amanhã a gente continua essa conversa.\\\"\\n  ],\\n  \\\"schema_version\\\": 2,\\n  \\\"kokoro_assets\\\": {\\n    \\\"release\\\": \\\"https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/\\\",\\n    \\\"files\\\": {\\n      \\\"kokoro-v1.0.int8.onnx\\\": \\\"6e742170d309016e5891a994e1ce1559c702a2ccd0075e67ef7157974f6406cb\\\",\\n      \\\"voices-v1.0.bin\\\": \\\"bca610b8308e8d99f32e6fe4197e7ec01679264efed0cac9140fe9c29f1fbf7d\\\",\\n      \\\"kokoro-v1.0.fp16-gpu.onnx\\\": \\\"c1610a859f3bdea01107e73e50100685af38fff88f5cd8e5c56df109ec880204\\\"\\n    },\\n    \\\"model_card_revision\\\": \\\"f3ff3571791e39611d31c381e3a41a3af07b4987\\\"\\n  }\\n}\", \"requirements-training.txt\": \"numpy==2.0.2\\nscipy==1.15.3\\npiper-tts==1.4.2\\nkokoro-onnx==0.4.9\\ntensorflow==2.18.1\\nkeras==3.15.1\\npymicro-features==2.0.2\\nmmap_ninja==0.9.0\\nPyYAML==6.0.3\\naudiomentations==0.43.1\\ndatasets==5.0.1\\nwebrtcvad-wheels==2.0.14.post1\\nai-edge-litert==2.2.0\\naudio-metadata @ git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f\\n\", \"test_lab.py\": \"import copy\\nimport json\\nfrom pathlib import Path\\nimport tempfile\\nimport unittest\\n\\nfrom lab import audit, read_config, training_config, sha256, summarize_scores\\nfrom run_upstream import ambient_guard\\nfrom colab_setup import run\\n\\n\\nclass DataProtectionTests(unittest.TestCase):\\n    def test_calibration_covers_saturated_scores_above_point_95(self):\\n        scores = [dict(split=s, positive=p, score=.996 if p else .99)\\n                  for s in ('validation', 'testing') for p in (True, False)]\\n        report = summarize_scores(scores)\\n        self.assertEqual(report['validation']['recall'], 1.)\\n        self.assertEqual(report['validation']['false_positive_rate'], 0.)\\n        self.assertGreater(report['validation']['threshold'], .99)\\n\\n    def test_resume_rejects_changed_settings_and_corrupted_audio(self):\\n        from diversity import read_progress\\n        with tempfile.TemporaryDirectory() as directory:\\n            work = Path(directory)\\n            audio = work / 'sample.wav'\\n            audio.write_bytes(b'original')\\n            path = work / 'progress.json'\\n            path.write_text(json.dumps(dict(signature='expected', completed=['voice'],\\n                records=[dict(path='sample.wav', sha256=sha256(audio))])))\\n            self.assertEqual(read_progress(path, 'expected', work)['completed'], ['voice'])\\n            with self.assertRaisesRegex(ValueError, 'parâmetros'):\\n                read_progress(path, 'changed', work)\\n            audio.write_bytes(b'corrupted')\\n            with self.assertRaisesRegex(ValueError, 'alterado'):\\n                read_progress(path, 'expected', work)\\n\\n    def test_cuda_failure_falls_back_to_cpu_without_claiming_gpu(self):\\n        from unittest.mock import Mock, patch\\n        from diversity import kokoro_session\\n        ort = Mock()\\n        ort.get_available_providers.return_value = ['CUDAExecutionProvider', 'CPUExecutionProvider']\\n        cpu = Mock()\\n        ort.InferenceSession.side_effect = [RuntimeError('missing CUDA library'), cpu]\\n        assets = dict(release='https://example.invalid/', files={\\n            'kokoro-v1.0.fp16-gpu.onnx': 'gpu', 'kokoro-v1.0.int8.onnx': 'cpu'})\\n        with patch('diversity.checked_asset'):\\n            session, digest = kokoro_session(ort, assets, Path('/test'), None)\\n        self.assertIs(session, cpu)\\n        self.assertEqual(digest, 'cpu')\\n        self.assertEqual(ort.InferenceSession.call_args.kwargs['providers'], ['CPUExecutionProvider'])\\n\\n    def test_allowed_false_positive_rate_does_not_force_rejection_of_all_calls(self):\\n        scores = []\\n        for split in ('validation', 'testing'):\\n            scores.extend(dict(split=split, positive=True, score=.8) for _ in range(10))\\n            scores.extend(dict(split=split, positive=False, score=.1) for _ in range(199))\\n            scores.append(dict(split=split, positive=False, score=.8))\\n        report = summarize_scores(scores)\\n        self.assertEqual(report['validation']['recall'], 1.)\\n        self.assertEqual(report['validation']['false_positive_rate'], .005)\\n\\n    def test_augmentation_preserves_whole_call_and_balances_duration(self):\\n        try:\\n            import numpy as np\\n            from diversity import augment\\n        except ImportError:\\n            self.skipTest('Teste de áudio exige o ambiente de treinamento.')\\n        call = np.sin(np.arange(16000) * .1) * .3\\n        for seed in range(10):\\n            positive = augment(call, np.random.default_rng(seed), True)\\n            negative = augment(call, np.random.default_rng(seed), False)\\n            self.assertEqual(positive.dtype, np.int16)\\n            self.assertEqual(len(positive), 52800)\\n            np.testing.assert_array_equal(positive, negative)\\n            # Too-long calls fail rather than silently training on a cut-off name.\\n        with self.assertRaisesRegex(ValueError, 'longa'):\\n            augment(np.ones(48000), np.random.default_rng(1), True)\\n\\n    def test_diverse_sources_include_native_and_female_heldout_profiles(self):\\n        config = read_config()\\n        self.assertGreaterEqual(len(config['voices']), 30)\\n        for split in ('training', 'validation', 'testing'):\\n            rows = [v for v in config['voices'] if v['split'] == split]\\n            self.assertTrue(any(v['native_pt'] for v in rows))\\n            self.assertTrue(any(v.get('style', '').startswith(('af_', 'bf_', 'pf_')) for v in rows))\\n        # Avoid known same-name language adaptations and the blended heart profile.\\n        self.assertFalse(any(v.get('style') in ('ef_dora', 'em_alex', 'em_santa', 'af_heart') for v in config['voices']))\\n\\n    def test_source_hash_mismatch_is_rejected(self):\\n        from diversity import checked_asset\\n        with tempfile.TemporaryDirectory() as directory:\\n            path = Path(directory) / 'model.onnx'\\n            path.write_bytes(b'wrong')\\n            with self.assertRaisesRegex(ValueError, 'Hash'):\\n                checked_asset('https://example.invalid/not-used', path, '0' * 64)\\n\\n    def test_diagnostics_export_only_known_model_files(self):\\n        from build_notebook import diagnostic_files\\n        with tempfile.TemporaryDirectory() as directory:\\n            work = Path(directory)\\n            trained = work / 'trained'\\n            trained.mkdir()\\n            for name in ('best_weights.weights.h5', 'private.wav', 'secret.json'):\\n                (trained / name).write_bytes(b'test')\\n            self.assertEqual(diagnostic_files(work), [trained / 'best_weights.weights.h5'])\\n\\n    def test_rejecting_every_call_is_a_failed_model(self):\\n        scores = [dict(split=s, positive=p, score=.54) for s in (\\\"validation\\\", \\\"testing\\\") for p in (True, False)]\\n        report = summarize_scores(scores)\\n        self.assertEqual(report[\\\"status\\\"], \\\"rejected_synthetic_evaluation\\\")\\n        self.assertEqual(report[\\\"diagnostics\\\"][\\\"testing\\\"][\\\"rank_auc\\\"], .5)\\n\\n    def test_test_speaker_cannot_change_selected_threshold(self):\\n        scores = [dict(split=s, positive=p, score=.8 if p else .1)\\n                  for s in (\\\"validation\\\", \\\"testing\\\") for p in (True, False)]\\n        threshold = summarize_scores(scores)[\\\"validation\\\"][\\\"threshold\\\"]\\n        scores[-2][\\\"score\\\"] = .3\\n        self.assertEqual(summarize_scores(scores)[\\\"validation\\\"][\\\"threshold\\\"], threshold)\\n        self.assertEqual(summarize_scores(scores)[\\\"testing\\\"][\\\"recall\\\"], 0)\\n\\n    def test_invalid_scores_cannot_pass_evaluation(self):\\n        with self.assertRaises(ValueError):\\n            summarize_scores([dict(split=\\\"validation\\\", positive=True, score=float(\\\"nan\\\"))])\\n    def rows(self):\\n        return [dict(voice=split, split=split, seconds=2., positive=positive)\\n                for split in (\\\"training\\\", \\\"validation\\\", \\\"testing\\\") for positive in (True, False)]\\n\\n    def test_distinct_speakers(self):\\n        audit(self.rows())\\n        rows = self.rows()\\n        rows[-1][\\\"voice\\\"] = \\\"training\\\"\\n        with self.assertRaisesRegex(ValueError, \\\"Vazamento\\\"):\\n            audit(rows)\\n\\n    def test_missing_negatives_cannot_be_validated(self):\\n        with self.assertRaises(ValueError):\\n            audit([r for r in self.rows() if r[\\\"positive\\\"]])\\n\\n    def test_unreviewed_license_is_rejected(self):\\n        config = read_config()\\n        config[\\\"voices\\\"][0][\\\"dataset_license\\\"] = \\\"CC-BY-NC-4.0\\\"\\n        with tempfile.TemporaryDirectory() as directory:\\n            path = Path(directory) / \\\"config.json\\\"\\n            path.write_text(json.dumps(config))\\n            with self.assertRaises(ValueError): read_config(path)\\n\\n    def test_duplicate_speaker_rejected(self):\\n        config = read_config()\\n        config[\\\"voices\\\"].append(copy.deepcopy(config[\\\"voices\\\"][0]))\\n        with tempfile.TemporaryDirectory() as directory:\\n            path = Path(directory) / \\\"config.json\\\"\\n            path.write_text(json.dumps(config))\\n            with self.assertRaises(ValueError): read_config(path)\\n\\n    def test_training_does_not_depend_on_absent_ambient_metric(self):\\n        config = training_config(Path(\\\"/experiment\\\"), 20)\\n        self.assertEqual(config[\\\"maximization_metric\\\"], \\\"recall\\\")\\n        self.assertEqual(config[\\\"minimization_metric\\\"], \\\"loss\\\")\\n        self.assertEqual(config[\\\"eval_step_interval\\\"], 20)\\n        self.assertNotIn(\\\"private\\\", json.dumps(config))\\n\\n    def test_quantization_calibration_has_whole_streaming_chunks(self):\\n        config = training_config(Path(\\\"/experiment\\\"), 20)\\n        output_frames = 1 + (config[\\\"clip_duration_ms\\\"] - 30) // 30\\n        dropped = 4 + (4 + 10 + 14 + 22) * 3\\n        self.assertEqual((output_frames + dropped) % 3, 0)\\n\\n    def test_no_ambient_data_is_not_reported_as_zero_false_accepts(self):\\n        from unittest.mock import Mock\\n        original = Mock(return_value=\\\"measured\\\")\\n        processor = Mock()\\n        processor.get_mode_size.return_value = 0\\n        self.assertIsNone(ambient_guard(original)({}, \\\"folder\\\", processor))\\n        original.assert_not_called()\\n        processor.get_mode_size.return_value = 2\\n        self.assertEqual(ambient_guard(original)({}, \\\"folder\\\", processor), \\\"measured\\\")\\n\\n    def test_audio_hash_changes_after_mutation(self):\\n        with tempfile.TemporaryDirectory() as directory:\\n            path = Path(directory) / \\\"sample.wav\\\"\\n            path.write_bytes(b\\\"before\\\")\\n            before = sha256(path)\\n            path.write_bytes(b\\\"after\\\")\\n            self.assertNotEqual(before, sha256(path))\\n\\n    def test_setup_error_preserves_real_stderr(self):\\n        import sys\\n        with tempfile.TemporaryDirectory() as directory:\\n            with self.assertRaisesRegex(RuntimeError, \\\"TEST_INSTALL_ERROR\\\"):\\n                run([sys.executable, \\\"-c\\\", \\\"import sys; print('TEST_INSTALL_ERROR', file=sys.stderr); sys.exit(7)\\\"],\\n                    Path(directory), \\\"Testar diagnóstico\\\")\\n            self.assertIn(\\\"TEST_INSTALL_ERROR\\\", (Path(directory) / \\\"setup.log\\\").read_text())\\n\\n\\nif __name__ == \\\"__main__\\\": unittest.main()\\n\", \"build_notebook.py\": \"\\\"\\\"\\\"Create a portable notebook from an explicit allowlist. No private audio/secrets.\\\"\\\"\\\"\\nimport json\\nfrom pathlib import Path\\n\\nROOT = Path(__file__).resolve().parent\\n\\n\\ndef diagnostic_files(work):\\n    \\\"\\\"\\\"Explicit model-only allowlist; never include checkpoints, audio or secrets.\\\"\\\"\\\"\\n    trained = work / 'trained'\\n    paths = [trained / name for name in (\\n        'best_weights.weights.h5', 'last_weights.weights.h5', 'training_config.yaml')]\\n    saved = trained / 'stream_state_internal'\\n    paths.extend(saved / name for name in (\\n        'saved_model.pb', 'fingerprint.pb', 'variables/variables.index',\\n        'variables/variables.data-00000-of-00001'))\\n    return [path for path in paths if path.is_file()]\\n\\n\\ndef build():\\n    cells = []\\n    def markdown(source):\\n        cells.append(dict(cell_type=\\\"markdown\\\", metadata={}, source=source.splitlines(True)))\\n    def code(source):\\n        cells.append(dict(cell_type=\\\"code\\\", metadata={}, execution_count=None, outputs=[], source=source.splitlines(True)))\\n    markdown(\\\"\\\"\\\"# Ei, Emily — protótipo microWakeWord em português\\n\\n**Experimento, não atualização pronta do Android.** Não requer suas gravações, API key ou plano pago.\\n\\n1. No Colab, escolha **Ambiente de execução → Alterar tipo → T4 GPU**, se disponível gratuitamente.\\n2. Escolha **Ambiente de execução → Executar tudo**. Confirme apenas a execução deste notebook.\\n3. No final baixe o ZIP e devolva ao desenvolvimento da Emily para avaliação.\\n\\nEsta segunda rodada usa **32 perfis sintéticos (Piper + Kokoro)**, incluindo femininos e masculinos. São sete perfis brasileiros e 25 estrangeiros sintetizados com fonemas pt-br; não são 32 falantes brasileiros reais. Perfis estrangeiros podem ter sotaque ou pronúncia inadequada; as prévias permitem conferir isso. Não usamos misturas de perfis para inflar o número de vozes.\\n\\nGera também frases parecidas, menções à Emily em conversas, silêncio e ruído. A duração mínima é balanceada entre classes, e a chamada inteira fica dentro da janela de treino. Vozes de validação/teste ficam separadas das de treino; não há garantia de independência de pessoas nos perfis internos do Kokoro. Resultados sintéticos não comprovam funcionamento para qualquer pessoa. Ainda faltam conversas reais longas, integração e consumo no Android. O seu áudio do WhatsApp não está incluído e não será enviado.\\n\\nO Colab gratuito pode interromper a sessão ou não oferecer GPU. Nenhum plano pago é ativado. A geração salva progresso por perfil e as etapas concluídas são reutilizadas enquanto os arquivos da sessão ainda existirem. Se o Colab apagar a máquina temporária, o progresso é perdido; não conectamos seu Google Drive automaticamente.\\n\\\"\\\"\\\")\\n    files = {name: (ROOT / name).read_text() for name in (\\\"lab.py\\\", \\\"diversity.py\\\", \\\"run_upstream.py\\\", \\\"colab_setup.py\\\", \\\"voices.json\\\", \\\"requirements-training.txt\\\", \\\"test_lab.py\\\", \\\"build_notebook.py\\\")}\\n    code(\\\"from pathlib import Path\\\\nimport json, os, subprocess, sys\\\\n\\\"\\n         \\\"root = Path('/content/emily-wakeword')\\\\nroot.mkdir(exist_ok=True)\\\\n\\\"\\n         f\\\"files = json.loads({json.dumps(json.dumps(files, ensure_ascii=False), ensure_ascii=False)})\\\\n\\\"\\n         \\\"for name, contents in files.items(): (root / name).write_text(contents)\\\\n\\\"\\n         \\\"os.chdir(root)\\\\n\\\"\\n         \\\"sys.path.insert(0, str(root))\\\\n\\\"\\n         \\\"os.environ['OMP_NUM_THREADS'] = '2'\\\\n\\\"\\n         \\\"os.environ['OPENBLAS_NUM_THREADS'] = '2'\\\\n\\\"\\n         \\\"os.environ['TF_NUM_INTRAOP_THREADS'] = '2'\\\\n\\\"\\n         \\\"os.environ['TF_NUM_INTEROP_THREADS'] = '1'\\\\n\\\"\\n         \\\"from colab_setup import setup\\\\n\\\"\\n         \\\"training_python = str(setup(root))\\\\n\\\")\\n    markdown(\\\"## Geração e treinamento\\\\nNão instale o resultado no celular ainda. A escolha do limiar usa somente validação; a voz de teste permanece separada.\\\")\\n    code(\\\"# Ajustáveis sem mexer na instalação da Emily.\\\\nREPETITIONS = 96\\\\nSTEPS = 6000\\\\n\\\"\\n         \\\"work = root / 'work-v2-diverse'\\\\n\\\"\\n         \\\"import hashlib, shutil\\\\n\\\"\\n         \\\"work.mkdir(exist_ok=True)\\\\n\\\"\\n         \\\"signature = hashlib.sha256(json.dumps({'files': files, 'repetitions': REPETITIONS, 'steps': STEPS}, sort_keys=True).encode()).hexdigest()\\\\n\\\"\\n         \\\"signature_path = work/'pipeline-signature.txt'\\\\n\\\"\\n         \\\"if signature_path.exists() and signature_path.read_text() != signature:\\\\n\\\"\\n         \\\"    raise RuntimeError('Pipeline alterado: use uma sessão nova para evitar misturar resultados.')\\\\n\\\"\\n         \\\"signature_path.write_text(signature)\\\\n\\\"\\n         \\\"for stage, options in [('generate', ['--repetitions', str(REPETITIONS)]), ('features', []), ('train', ['--steps', str(STEPS)]), ('evaluate', [])]:\\\\n\\\"\\n         \\\"    done = work/(stage+'.done')\\\\n\\\"\\n         \\\"    if done.is_file():\\\\n\\\"\\n         \\\"        print('Etapa já concluída:', stage, flush=True)\\\\n\\\"\\n         \\\"        continue\\\\n\\\"\\n         \\\"    if stage == 'features' and (work/'features').exists():\\\\n\\\"\\n         \\\"        shutil.rmtree(work/'features')  # Apenas derivados incompletos desta sessão isolada.\\\\n\\\"\\n         \\\"    print('Etapa:', stage, flush=True)\\\\n\\\"\\n         \\\"    subprocess.run([training_python, 'lab.py', stage, '--work', str(work), *options], check=True)\\\\n\\\"\\n         \\\"    done.write_text(signature)\\\\n\\\"\\n         \\\"    if stage == 'generate':\\\\n\\\"\\n         \\\"        from IPython.display import Audio, display\\\\n\\\"\\n         \\\"        rows = [json.loads(line) for line in (work/'samples.jsonl').read_text().splitlines()]\\\\n\\\"\\n         \\\"        for voice in json.loads(Path('voices.json').read_text())['voices']:\\\\n\\\"\\n         \\\"            example = next(r for r in rows if r['voice'] == voice['id'] and r['positive'])\\\\n\\\"\\n         \\\"            print('Amostra de pronúncia:', voice['id'])\\\\n\\\"\\n         \\\"            display(Audio(filename=str(work/example['path'])))\\\\n\\\")\\n    markdown(\\\"## Resultado para avaliação\\\\nO ZIP contém o modelo candidato, pesos e modelo original para comparar a conversão, relatório, origens/licenças e hashes. Não inclui gravações suas, arquivos da Emily ou vozes completas do Piper. A avaliação sintética não é autorização para distribuir o detector.\\\")\\n    code(\\\"import zipfile\\\\nfrom google.colab import files as colab_files\\\\n\\\"\\n         \\\"from build_notebook import diagnostic_files\\\\n\\\"\\n         \\\"result = root / 'ei-emily-diverse-v2-candidate.zip'\\\\n\\\"\\n         \\\"model = work / 'trained/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite'\\\\n\\\"\\n         \\\"with zipfile.ZipFile(result, 'w', zipfile.ZIP_DEFLATED) as archive:\\\\n\\\"\\n         \\\"    for path in [model, work/'report.json', work/'resolved_sources.json', work/'samples.jsonl', work/'training.yaml']:\\\\n\\\"\\n         \\\"        archive.write(path, str(path.relative_to(work)))\\\\n\\\"\\n         \\\"    for path in diagnostic_files(work):\\\\n\\\"\\n         \\\"        archive.write(path, str(path.relative_to(work)))\\\\n\\\"\\n         \\\"    for path in (work/'voices').glob('*/MODEL_CARD'):\\\\n\\\"\\n         \\\"        archive.write(path, str(path.relative_to(work)))\\\\n\\\"\\n         \\\"    for name in ('README.md', 'VOICES.md'):\\\\n\\\"\\n         \\\"        path = work/'kokoro'/name\\\\n\\\"\\n         \\\"        if path.is_file(): archive.write(path, str(path.relative_to(work)))\\\\n\\\"\\n         \\\"    archive.write('upstream/LICENSE', 'licenses/microwakeword-APACHE-2.0.txt')\\\\n\\\"\\n         \\\"print((work/'report.json').read_text())\\\\ncolab_files.download(str(result))\\\\n\\\")\\n    result = dict(nbformat=4, nbformat_minor=5, cells=cells,\\n                  metadata=dict(kernelspec=dict(display_name=\\\"Python 3\\\", language=\\\"python\\\", name=\\\"python3\\\"),\\n                                language_info=dict(name=\\\"python\\\"), colab=dict(name=\\\"ei-emily-colab.ipynb\\\")))\\n    for index, cell in enumerate(cells): cell[\\\"id\\\"] = f\\\"emily-{index}\\\"\\n    destination = ROOT / \\\"ei-emily-colab.ipynb\\\"\\n    destination.write_text(json.dumps(result, indent=1, ensure_ascii=False))\\n    for cell in cells:\\n        if cell[\\\"cell_type\\\"] == \\\"code\\\": compile(\\\"\\\".join(cell[\\\"source\\\"]), str(destination), \\\"exec\\\")\\n    print(destination)\\n\\n\\nif __name__ == \\\"__main__\\\": build()\\n\"}")
for name, contents in files.items(): (root / name).write_text(contents)
os.chdir(root)
sys.path.insert(0, str(root))
os.environ['OMP_NUM_THREADS'] = '2'
os.environ['OPENBLAS_NUM_THREADS'] = '2'
os.environ['TF_NUM_INTRAOP_THREADS'] = '2'
os.environ['TF_NUM_INTEROP_THREADS'] = '1'
from colab_setup import setup
training_python = str(setup(root))


## Geração e treinamento
Não instale o resultado no celular ainda. A escolha do limiar usa somente validação; a voz de teste permanece separada.

In [ ]:
# Ajustáveis sem mexer na instalação da Emily.
REPETITIONS = 96
STEPS = 6000
work = root / 'work-v2-diverse'
import hashlib, shutil
work.mkdir(exist_ok=True)
signature = hashlib.sha256(json.dumps({'files': files, 'repetitions': REPETITIONS, 'steps': STEPS}, sort_keys=True).encode()).hexdigest()
signature_path = work/'pipeline-signature.txt'
if signature_path.exists() and signature_path.read_text() != signature:
    raise RuntimeError('Pipeline alterado: use uma sessão nova para evitar misturar resultados.')
signature_path.write_text(signature)
for stage, options in [('generate', ['--repetitions', str(REPETITIONS)]), ('features', []), ('train', ['--steps', str(STEPS)]), ('evaluate', [])]:
    done = work/(stage+'.done')
    if done.is_file():
        print('Etapa já concluída:', stage, flush=True)
        continue
    if stage == 'features' and (work/'features').exists():
        shutil.rmtree(work/'features')  # Apenas derivados incompletos desta sessão isolada.
    print('Etapa:', stage, flush=True)
    subprocess.run([training_python, 'lab.py', stage, '--work', str(work), *options], check=True)
    done.write_text(signature)
    if stage == 'generate':
        from IPython.display import Audio, display
        rows = [json.loads(line) for line in (work/'samples.jsonl').read_text().splitlines()]
        for voice in json.loads(Path('voices.json').read_text())['voices']:
            example = next(r for r in rows if r['voice'] == voice['id'] and r['positive'])
            print('Amostra de pronúncia:', voice['id'])
            display(Audio(filename=str(work/example['path'])))


## Resultado para avaliação
O ZIP contém o modelo candidato, pesos e modelo original para comparar a conversão, relatório, origens/licenças e hashes. Não inclui gravações suas, arquivos da Emily ou vozes completas do Piper. A avaliação sintética não é autorização para distribuir o detector.

In [ ]:
import zipfile
from google.colab import files as colab_files
from build_notebook import diagnostic_files
result = root / 'ei-emily-diverse-v2-candidate.zip'
model = work / 'trained/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite'
with zipfile.ZipFile(result, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in [model, work/'report.json', work/'resolved_sources.json', work/'samples.jsonl', work/'training.yaml']:
        archive.write(path, str(path.relative_to(work)))
    for path in diagnostic_files(work):
        archive.write(path, str(path.relative_to(work)))
    for path in (work/'voices').glob('*/MODEL_CARD'):
        archive.write(path, str(path.relative_to(work)))
    for name in ('README.md', 'VOICES.md'):
        path = work/'kokoro'/name
        if path.is_file(): archive.write(path, str(path.relative_to(work)))
    archive.write('upstream/LICENSE', 'licenses/microwakeword-APACHE-2.0.txt')
print((work/'report.json').read_text())
colab_files.download(str(result))
